In [4]:
import pandas as pd

In [7]:
import pandas as pd

# Adding the encoding parameter fixes the Unicode error
df = pd.read_csv('data.csv', encoding='unicode_escape')

# Display the first few rows to check if it worked
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


In [8]:
print("Original Rows:",len(df))

Original Rows: 541909


In [9]:
df_small=df.sample(n=10000,random_state=42)

In [10]:
print("Reduced Rows:",len(df_small))

Reduced Rows: 10000


In [11]:
df_small=df_small.reset_index(drop=True)

In [12]:
df_small.to_csv('data_small.csv', index=False)

In [14]:
import pandas as pd

# 1. Load the newly created small dataset
print("Loading data_small.csv...")
# We still use unicode_escape just in case any British Pound symbols (£) are in these 10,000 rows
df = pd.read_csv('data_small.csv', encoding='unicode_escape')
print(f"Starting shape: {df.shape}")

# 2. Data Cleaning & Preprocessing
print("Cleaning data...")
# Drop rows where CustomerID is missing
df_clean = df.dropna(subset=['CustomerID']).copy()

# Standardize CustomerID format from decimal to string
df_clean['CustomerID'] = df_clean['CustomerID'].astype(int).astype(str)

# Convert InvoiceDate to a proper Datetime format
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])

# Handle anomalies: Keep only rows with positive quantities and prices (removes returns/errors)
df_clean = df_clean[(df_clean['Quantity'] > 0) & (df_clean['UnitPrice'] > 0)]
print(f"Cleaned shape: {df_clean.shape}")

# 3. Feature Engineering
print("Creating new features...")
# Create TotalSales metric
df_clean['TotalSales'] = df_clean['Quantity'] * df_clean['UnitPrice']

# Extract time-based features for dashboard charts
df_clean['Month'] = df_clean['InvoiceDate'].dt.month_name()
df_clean['DayOfWeek'] = df_clean['InvoiceDate'].dt.day_name()
df_clean['Hour'] = df_clean['InvoiceDate'].dt.hour

# 4. Exploratory Data Analysis (KPIs)
print("\n--- Exploratory Data Analysis ---")
total_revenue = df_clean['TotalSales'].sum()
total_orders = df_clean['InvoiceNo'].nunique()
unique_customers = df_clean['CustomerID'].nunique()

print(f"Total Revenue: ${total_revenue:,.2f}")
print(f"Total Successful Orders: {total_orders}")
print(f"Total Unique Customers: {unique_customers}")

print("\nTop 5 Products by Revenue:")
# Group by product description, sum the sales, sort highest to lowest, and show top 5
top_products = df_clean.groupby('Description')['TotalSales'].sum().sort_values(ascending=False).head(5)
print(top_products)

# 5. Export for Dashboarding
df_clean.to_csv('ecommerce_ready_for_dashboard.csv', index=False)
print("\nSuccess! Data saved as 'ecommerce_ready_for_dashboard.csv'. Ready for visualization!")

Loading data_small.csv...
Starting shape: (10000, 8)
Cleaning data...
Cleaned shape: (7323, 8)
Creating new features...

--- Exploratory Data Analysis ---
Total Revenue: $152,790.36
Total Successful Orders: 5211
Total Unique Customers: 2405

Top 5 Products by Revenue:
Description
GIN + TONIC DIET METAL SIGN    3629.39
JUMBO BAG RED RETROSPOT        2931.50
JUMBO BAG STRAWBERRY           1787.17
ALARM CLOCK BAKELIKE PINK      1709.25
FELTCRAFT DOLL MOLLY           1706.15
Name: TotalSales, dtype: float64

Success! Data saved as 'ecommerce_ready_for_dashboard.csv'. Ready for visualization!
